In [20]:
import pyGinkgo as pg

### Direct solver bindings

In [21]:
fn = "m1.mtx"
executor = pg.device("cuda")
mtx = pg.read(path=fn, dtype="double", format="Csr")
n_rows = mtx.shape[0]

b = pg.dense(dim=(n_rows, 1), device=executor, dtype="double", fill=1.0)
x = pg.dense(dim=(n_rows, 1), device=executor, dtype="double", fill=0.0)

# Create ILU preconditioner
preconditioner = pg.preconditioner.Ilu(executor, mtx)

# Setup GMRES solver
solver = pg.solver.gmres(
    executor,
    mtx,
    preconditioner=preconditioner,
    max_iters=1000,
    krylov_dim=30,
    reduction_factor=1e-06,
)

# Apply
logger, result = solver.apply(b, x)

In [22]:
result_cpu = result.copy_to_host()

/home/mouad/dev/pyGinkgo-dev/pyGinkgo/src/cpp_bindings/dense.cppWarning creating a copy of dense


In [23]:
for i in range(10):
    print(result_cpu.at(i, 0), end=", ")

0.6044527378669796, 0.8250277658127815, 0.9154476498978679, 0.9545267011444474, 0.9719330029987087, 0.9798380401260545, 0.9834770837858334, 0.9851691223643666, 0.985961924621025, 0.9863356545760611, 

### Config solver

In [24]:
args = {
    "type": "solver::Gmres",
    "krylov_dim": 30,
    "preconditioner": {
        "type": "preconditioner::Jacobi",
        "max_block_size": 1
    },
    "criteria": [
        {"type": "Iteration", "max_iters": 1000},
        {
          "type": "ResidualNorm", 
          "reduction_factor": 1e-6, 
          "baseline": "rhs_norm"
        }
    ],
}

In [25]:
b = pg.dense(dim=(n_rows, 1), device=executor, dtype="double", fill=1.0)
x = pg.dense(dim=(n_rows, 1), device=executor, dtype="double", fill=0.0)

solver = pg.generate_solver(mtx, args)

logger, result = solver.apply(b, x)

In [26]:
result_cpu_2 = result.copy_to_host()

/home/mouad/dev/pyGinkgo-dev/pyGinkgo/src/cpp_bindings/dense.cppWarning creating a copy of dense


### Checking difference between two solvers

In [27]:
result_cpu_2.add_scaled(
    pg.dense(dim=(1, 1), device=executor, dtype="double", fill=-1),
    result_cpu
)

In [28]:
for i in range(10):
    print(result_cpu_2.at(i, 0), end=", ")

2.362639306419112e-09, -5.442144401790472e-09, -2.802585497008181e-09, -1.7956988895839743e-08, -4.789233010171756e-09, -2.7631512966586058e-08, -5.420423110358286e-09, -3.681777438480083e-08, -1.045112496544931e-08, -5.73117376889698e-08, 